# 🧮 Vector Embeddings và Đo lường Độ Tương Đồng Ngữ Nghĩa

## Mục tiêu bài học
- Hiểu khái niệm Vector Embeddings và cách biểu diễn ngữ nghĩa trong không gian vector
- Nắm vững các phương pháp đo lường độ tương đồng:
  - Cosine Similarity
  - Euclidean Distance
  - Dot Product
- Áp dụng vào bài toán thực tế với văn bản tiếng Việt và tiếng Anh
- So sánh ưu nhược điểm của từng phương pháp

## 🛠️ Phần 2: Cài đặt và Chuẩn bị

In [1]:
# Cài đặt các thư viện cần thiết
%pip install sentence-transformers numpy matplotlib scikit-learn pandas gensim


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/24.4 MB ? eta -:--:--
   ---------------------------------------- 0.1/24.4 MB 1.3 MB/s eta 0:00:19
   ---------------------------------------- 0.1/24.4 MB 1.4 MB/s eta 0:00:17
   ---------------------------------------- 0.3/24.4 MB 2.0 MB/s eta 0:00:13
    --------------------------------------- 0.5/24.4 MB 2.7 MB/s eta 0:00:09
   - -------------------------------------- 0.7/24.4 MB 3.1 MB/s eta 0:00:08
   - -------------------------------------- 1.1/24.4 MB 4.3 MB/s eta 0:00:06
   --- ------------------------------------ 2.2/24.4 MB 6.7 MB/s eta 0:00:04
   ------ --------------------------------- 3.7/24.4 MB 9.9 MB/s eta 0:00:03
   ---------- ----------------------------- 6.3/24.4 MB 15.0 MB/s eta 0:00:02
   ---------------- ----------------------- 9.8/24.4 MB 21.0 MB/s eta 0:00:01
   ----------------------- ---------------- 14.0/24.4 MB 73.1 MB/s eta 0:00:01
   --------------------------- ------------ 16.9/24.4 MB 81.8 MB/s eta 0:00:01


In [1]:
# Import thư viện
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

### Load Pre-trained Embedding Model

Chúng ta sẽ sử dụng **sentence-transformers**, một thư viện mạnh mẽ cho embeddings:
- Model: `paraphrase-multilingual-MiniLM-L12-v2` - hỗ trợ 50+ ngôn ngữ, bao gồm tiếng Việt
- Kích thước vector: 384 chiều
- Nhanh và hiệu quả cho hầu hết các ứng dụng

In [2]:
# Load model đa ngôn ngữ
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print(f"✅ Model loaded successfully!")
print(f"📊 Embedding dimension: {model.get_sentence_embedding_dimension()}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Model loaded successfully!
📊 Embedding dimension: 384


## 🔬 Phần 3: Tạo Vector Embeddings

### 3.1 Embeddings cho từ và câu đơn giản

In [3]:
# Tạo embeddings cho các câu tiếng Anh
sentences_en = [
    "I love programming",
    "I enjoy coding",
    "The weather is nice today",
    "Machine learning is fascinating",
    "Deep learning is a subset of AI"
]

# Tạo embeddings
embeddings_en = model.encode(sentences_en)

print("=" * 70)
print("EMBEDDINGS TIẾNG ANH")
print("=" * 70)
for i, sentence in enumerate(sentences_en):
    print(f"\nCâu {i+1}: {sentence}")
    print(f"First 10 values: {embeddings_en[i][:10]}")
    print(f"Vector norm: {np.linalg.norm(embeddings_en[i]):.4f}")

EMBEDDINGS TIẾNG ANH

Câu 1: I love programming
First 10 values: [-0.09754957 -0.37010992 -0.04942968 -0.11293895 -0.24742271 -0.15186767
  0.0716017   0.17737548  0.0095045   0.5964115 ]
Vector norm: 5.6580

Câu 2: I enjoy coding
First 10 values: [ 0.05587781 -0.4196714  -0.2494934   0.00381995 -0.311845    0.30276927
  0.30298117  0.27343205 -0.21720481  0.7214778 ]
Vector norm: 5.5543

Câu 3: The weather is nice today
First 10 values: [ 0.28088745  0.27709776 -0.07001772  0.08150014  0.24186812  0.09056901
  0.5559215  -0.10870065 -0.58855444  0.13057   ]
Vector norm: 4.9209

Câu 4: Machine learning is fascinating
First 10 values: [-0.1311065  -0.15423486 -0.27258596 -0.32290596 -0.09494047 -0.10874902
 -0.06425487 -0.00107528 -0.03084948 -0.08673868]
Vector norm: 4.9938

Câu 5: Deep learning is a subset of AI
First 10 values: [-0.1499104  -0.17846772 -0.0457218   0.00266017 -0.04144255  0.3145865
 -0.00095411 -0.27811605  0.27148744 -0.08658599]
Vector norm: 4.6359


In [13]:
# Tạo embeddings cho các câu tiếng Việt
sentences_vi = [
    "Tôi yêu lập trình",
    "Tôi thích viết code",
    "Hôm nay thời tiết đẹp",
    "Học máy rất hấp dẫn",
    "Học sâu là một phần của AI"
]

embeddings_vi = model.encode(sentences_vi)

print("=" * 70)
print("EMBEDDINGS TIẾNG VIỆT")
print("=" * 70)
for i, sentence in enumerate(sentences_vi):
    print(f"\nCâu {i+1}: {sentence}")
    print(f"First 10 values: {embeddings_vi[i][:10]}")
    print(f"Vector norm: {np.linalg.norm(embeddings_vi[i]):.4f}")

EMBEDDINGS TIẾNG VIỆT

Câu 1: Tôi yêu lập trình
First 10 values: [-0.075303    0.04626438 -0.02472326  0.02352159 -0.37955198 -0.17767237
  0.31471893  0.14991693  0.16120149  0.06134453]
Vector norm: 3.5849

Câu 2: Tôi thích viết code
First 10 values: [-0.09226795  0.06693625 -0.32328966 -0.13070191 -0.2620339  -0.04802443
  0.06542933  0.08910932 -0.05534301  0.18100159]
Vector norm: 3.6425

Câu 3: Hôm nay thời tiết đẹp
First 10 values: [ 0.15501285  0.25530383  0.03599243 -0.06592956 -0.37851045 -0.11839836
  0.2175422   0.22812256 -0.04627054  0.05023238]
Vector norm: 4.9620

Câu 4: Học máy rất hấp dẫn
First 10 values: [ 0.00254175 -0.02505935 -0.1808393  -0.03892576 -0.45883825 -0.14935172
  0.2890493   0.2767208   0.15175703 -0.06403521]
Vector norm: 4.0595

Câu 5: Học sâu là một phần của AI
First 10 values: [ 0.00648856  0.16515297 -0.02012777  0.04519362 -0.4117417  -0.2762395
  0.4381302   0.12934002  0.08363736  0.01565728]
Vector norm: 3.8775


### Static vs Contextual embedding 


In [2]:
import numpy as np
import gensim.downloader as api

print("🔄 Loading GloVe model (static embedding)...")
glove = api.load('glove-wiki-gigaword-50')  # 50-dim, ~66MB
print("✅ GloVe loaded!\n")

# Static embedding: "bank" trong ngữ cảnh tài chính và tự nhiên
# → GloVe chỉ là lookup table, không quan tâm câu xung quanh
sentence_finance = "I went to the bank to deposit my money"
sentence_nature  = "The river bank was covered with green grass"

# Dù 2 câu hoàn toàn khác nghĩa, GloVe vẫn trả về CÙNG 1 VECTOR cho "bank"
static_vec_finance = glove["bank"]  # vector của "bank" trong câu tài chính
static_vec_nature  = glove["bank"]  # vector của "bank" trong câu tự nhiên → giống hệt

print("📌 STATIC EMBEDDING (GloVe)")
print(f"\nCâu 1: '{sentence_finance}'")
print(f"Vector 'bank' (10 dim đầu): {static_vec_finance[:10].round(4)}")

print(f"\nCâu 2: '{sentence_nature}'")
print(f"Vector 'bank' (10 dim đầu): {static_vec_nature[:10].round(4)}")

print(f"\n⚠️  Hai vector hoàn toàn GIỐNG HỆT nhau vì GloVe không đọc ngữ cảnh câu")

🔄 Loading GloVe model (static embedding)...
[==================================================] 100.0% 66.0/66.0MB downloaded
✅ GloVe loaded!

📌 STATIC EMBEDDING (GloVe)
Từ: 'bank'
Vector (10 dim đầu): [ 0.6649 -0.1139  0.6784  0.1795  0.6828 -0.4779 -0.3076  0.1749 -0.7051
 -0.5502]

⚠️  'bank' (ngân hàng) và 'bank' (bờ sông) → CÙNG 1 VECTOR


In [3]:
from transformers import AutoTokenizer, AutoModel
import torch

print("🔄 Loading BERT model (contextual embedding)...")
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
bert_model = AutoModel.from_pretrained("distilbert-base-uncased")
bert_model.eval()
print("✅ BERT loaded!\n")

def get_word_vector_in_context(sentence, target_word, tokenizer, model):
    """Lấy vector của target_word trong ngữ cảnh câu (contextual)."""
    inputs = tokenizer(sentence, return_tensors="pt")
    tokens = tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

    with torch.no_grad():
        hidden = model(**inputs).last_hidden_state[0]  # (seq_len, 768)

    # Tìm index các sub-tokens của target_word
    indices = [i for i, t in enumerate(tokens) if target_word.lower() in t.lower()]
    if not indices:
        raise ValueError(f"'{target_word}' không tìm thấy trong tokens: {tokens}")

    # Average nếu target_word bị tách thành nhiều sub-tokens
    return hidden[indices].mean(dim=0).numpy()

# Cùng từ "bank" — 2 nghĩa hoàn toàn khác nhau
sentence_finance = "I went to the bank to deposit my money"
sentence_nature  = "The river bank was covered with green grass"

vec_finance = get_word_vector_in_context(sentence_finance, "bank", tokenizer, bert_model)
vec_nature  = get_word_vector_in_context(sentence_nature,  "bank", tokenizer, bert_model)

print("📌 CONTEXTUAL EMBEDDING (BERT)")
print(f"\nCâu 1: '{sentence_finance}'")
print(f"Vector 'bank' (10 dim đầu): {vec_finance[:10].round(4)}")

print(f"\nCâu 2: '{sentence_nature}'")
print(f"Vector 'bank' (10 dim đầu): {vec_nature[:10].round(4)}")

🔄 Loading BERT model (contextual embedding)...


config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ BERT loaded!

📌 CONTEXTUAL EMBEDDING (BERT)

Câu 1: 'I went to the bank to deposit my money'
Vector 'bank' (10 dim đầu): [ 0.2313 -0.0412 -0.0569  0.0196  0.7849 -0.0544 -0.2636  0.764  -0.2974
  0.2072]

Câu 2: 'The river bank was covered with green grass'
Vector 'bank' (10 dim đầu): [-0.0634 -0.2577 -0.1702 -0.0874  0.258   0.4441 -0.1707  0.995   0.105
 -0.2856]


In [4]:
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

# Static: 2 câu khác nghĩa nhưng cùng từ "bank" → cùng vector → similarity = 1.0
static_sim     = cosine_sim(static_vec_finance, static_vec_nature)
# Contextual: BERT tạo vector khác nhau theo ngữ cảnh → similarity < 1.0
contextual_sim = cosine_sim(vec_finance, vec_nature)

print("=" * 60)
print("📊 KẾT QUẢ SO SÁNH")
print("=" * 60)
print(f"\nCâu 1: 'I went to the BANK to deposit my money'  (ngân hàng)")
print(f"Câu 2: 'The river BANK was covered with green grass'  (bờ sông)")

print(f"\n🔵 STATIC (GloVe)")
print(f"   Cosine Similarity: {static_sim:.4f}  ← = 1.0, không phân biệt được 2 nghĩa")

print(f"\n🟢 CONTEXTUAL (BERT)")
print(f"   Cosine Similarity: {contextual_sim:.4f}  ← < 1.0, BERT phân biệt được 2 nghĩa")

print(f"\n💡 NHẬN XÉT:")
print(f"   Static     → lookup table cố định: 'bank' luôn = 1 vector duy nhất")
print(f"   Contextual → Transformer đọc toàn bộ câu: vector thay đổi theo ngữ cảnh")
print(f"   Similarity BERT = {contextual_sim:.4f} → 2 nghĩa của 'bank' đã được phân tách")

📊 KẾT QUẢ SO SÁNH

🔵 STATIC (GloVe)
   'bank' (ngân hàng) vs 'bank' (bờ sông)
   Cosine Similarity: 1.0000  ← luôn = 1.0, không phân biệt ngữ cảnh

🟢 CONTEXTUAL (BERT)
   'bank' (ngân hàng) vs 'bank' (bờ sông)
   Cosine Similarity: 0.6773  ← nhỏ hơn 1.0, BERT phân biệt được 2 nghĩa

💡 NHẬN XÉT:
   Static  → mỗi từ có đúng 1 vector cố định (lookup table)
   Contextual → vector thay đổi theo ngữ cảnh câu xung quanh
   Similarity BERT = 0.6773 → 2 nghĩa của 'bank' đã được phân tách


## 📐 Phần 4: Các Phép Đo Độ Tương Đồng

### 4.1 Cosine Similarity (Độ tương đồng Cosine)

**Công thức:**
$$\text{cosine\_similarity}(A, B) = \frac{A \cdot B}{\|A\| \times \|B\|} = \frac{\sum_{i=1}^{n} A_i \times B_i}{\sqrt{\sum_{i=1}^{n} A_i^2} \times \sqrt{\sum_{i=1}^{n} B_i^2}}$$

**Giá trị:**
- Range: [-1, 1]
- 1: Hoàn toàn giống nhau (cùng hướng)
- 0: Không liên quan (vuông góc)
- -1: Hoàn toàn ngược nhau

**Đặc điểm:**
- ✅ Không phụ thuộc vào độ lớn của vector
- ✅ Chỉ quan tâm đến hướng/góc giữa các vector
- ✅ Phổ biến nhất trong NLP và Recommendation Systems
- ❌ Không tính đến magnitude

In [4]:
# Test với 2 câu đầu tiên (tiếng Anh)
vec1 = embeddings_en[0]  # "I love programming"
vec2 = embeddings_en[1]  # "I enjoy coding"
vec3 = embeddings_en[2]  # "The weather is nice today"

In [6]:
# Implement Cosine Similarity từ đầu
def cosine_similarity_manual(vec1, vec2):
    """
    Tính cosine similarity giữa 2 vectors
    """
    dot_product = np.dot(vec1, vec2)
    norm_vec1 = np.linalg.norm(vec1)
    norm_vec2 = np.linalg.norm(vec2)
    
    return dot_product / (norm_vec1 * norm_vec2)



print("=" * 70)
print("COSINE SIMILARITY - TIẾNG ANH")
print("=" * 70)

sim_1_2 = cosine_similarity_manual(vec1, vec2)
sim_1_3 = cosine_similarity_manual(vec1, vec3)

print(f"\n'{sentences_en[0]}' vs '{sentences_en[1]}'")
print(f"Cosine Similarity: {sim_1_2:.4f}")

print(f"\n'{sentences_en[0]}' vs '{sentences_en[2]}'")
print(f"Cosine Similarity: {sim_1_3:.4f}")

print(f"\n💡 Nhận xét: Câu 1 và 2 có ý nghĩa tương tự → similarity cao ({sim_1_2:.4f})")
print(f"💡 Nhận xét: Câu 1 và 3 khác nghĩa → similarity thấp ({sim_1_3:.4f})")

COSINE SIMILARITY - TIẾNG ANH

'I love programming' vs 'I enjoy coding'
Cosine Similarity: 0.8598

'I love programming' vs 'The weather is nice today'
Cosine Similarity: 0.1603

💡 Nhận xét: Câu 1 và 2 có ý nghĩa tương tự → similarity cao (0.8598)
💡 Nhận xét: Câu 1 và 3 khác nghĩa → similarity thấp (0.1603)


### 4.2 Euclidean Distance (Khoảng cách Euclidean)

**Công thức:**
$$\text{euclidean\_distance}(A, B) = \sqrt{\sum_{i=1}^{n} (A_i - B_i)^2}$$

**Giá trị:**
- Range: [0, ∞]
- 0: Hai vector giống hệt nhau
- Càng lớn: Càng khác nhau

**Đặc điểm:**
- ✅ Đơn giản, trực quan
- ✅ Tính đến cả magnitude và direction
- ❌ Phụ thuộc vào độ lớn của vector
- ❌ Nhạy cảm với scale của dữ liệu
- ❌ Không hiệu quả với high-dimensional data (curse of dimensionality)

In [7]:
# Implement Euclidean Distance từ đầu
def euclidean_distance_manual(vec1, vec2):
    """
    Tính Euclidean distance giữa 2 vectors
    """
    return np.sqrt(np.sum((vec1 - vec2) ** 2))

print("=" * 70)
print("EUCLIDEAN DISTANCE - TIẾNG ANH")
print("=" * 70)

dist_1_2 = euclidean_distance_manual(vec1, vec2)
dist_1_3 = euclidean_distance_manual(vec1, vec3)

print(f"\n'{sentences_en[0]}' vs '{sentences_en[1]}'")
print(f"Euclidean Distance: {dist_1_2:.4f}")

print(f"\n'{sentences_en[0]}' vs '{sentences_en[2]}'")
print(f"Euclidean Distance: {dist_1_3:.4f}")

print(f"\n💡 Nhận xét: Câu 1 và 2 tương tự → distance nhỏ ({dist_1_2:.4f})")
print(f"💡 Nhận xét: Câu 1 và 3 khác nhau → distance lớn ({dist_1_3:.4f})")

EUCLIDEAN DISTANCE - TIẾNG ANH

'I love programming' vs 'I enjoy coding'
Euclidean Distance: 2.9699

'I love programming' vs 'The weather is nice today'
Euclidean Distance: 6.8776

💡 Nhận xét: Câu 1 và 2 tương tự → distance nhỏ (2.9699)
💡 Nhận xét: Câu 1 và 3 khác nhau → distance lớn (6.8776)


#### 💡 Thử nghiệm với câu của bạn!

Bạn có thể thay đổi 2 câu tiếng Việt bất kỳ để xem khoảng cách Euclidean giữa chúng:

In [ ]:
# TODO: Thay đổi 2 câu của bạn ở đây
cau_1 = "Nhập câu thứ nhất của bạn..."
cau_2 = "Nhập câu thứ hai của bạn..."

# Uncomment dòng dưới để chạy
# result = visualize_euclidean_distance(cau_1, cau_2, model, language='vi')

### 4.3 Dot Product (Tích vô hướng)

**Công thức:**
$$\text{dot\_product}(A, B) = \sum_{i=1}^{n} A_i \times B_i = A_1 \times B_1 + A_2 \times B_2 + ... + A_n \times B_n$$

**Giá trị:**
- Range: (-∞, ∞)
- Càng lớn: Càng tương đồng
- Giá trị âm: Vectors ngược hướng

**Đặc điểm:**
- ✅ Tính toán đơn giản, nhanh nhất
- ✅ Tính đến cả magnitude và direction
- ✅ Hiệu quả cho normalized vectors
- ❌ Phụ thuộc vào độ lớn của vector
- ❌ Khó so sánh giữa các cặp vectors khác nhau

**Lưu ý:** Nếu vectors được normalize (unit vectors), dot product = cosine similarity!

In [13]:
# Implement Dot Product từ đầu
def dot_product_manual(vec1, vec2):
    """
    Tính dot product giữa 2 vectors
    """
    return np.dot(vec1, vec2)

print("=" * 70)
print("DOT PRODUCT - TIẾNG ANH")
print("=" * 70)

dp_1_2 = dot_product_manual(vec1, vec2)
dp_1_3 = dot_product_manual(vec1, vec3)

print(f"\n'{sentences_en[0]}' vs '{sentences_en[1]}'")
print(f"Dot Product: {dp_1_2:.4f}")

print(f"\n'{sentences_en[0]}' vs '{sentences_en[2]}'")
print(f"Dot Product: {dp_1_3:.4f}")

print(f"\n💡 Nhận xét: Câu 1 và 2 tương tự → dot product cao ({dp_1_2:.4f})")
print(f"💡 Nhận xét: Câu 1 và 3 khác nhau → dot product thấp hơn ({dp_1_3:.4f})")

DOT PRODUCT - TIẾNG ANH

'I love programming' vs 'I enjoy coding'
Dot Product: 27.0219

'I love programming' vs 'The weather is nice today'
Dot Product: 4.4634

💡 Nhận xét: Câu 1 và 2 tương tự → dot product cao (27.0219)
💡 Nhận xét: Câu 1 và 3 khác nhau → dot product thấp hơn (4.4634)


### 5.3 So sánh tiếng Việt và tiếng Anh

In [ ]:
# So sánh cross-language: "I love programming" (EN) vs "Tôi yêu lập trình" (VI)
vec_en = embeddings_en[0]
vec_vi = embeddings_vi[0]

cos_sim_cross = cosine_similarity_manual(vec_en, vec_vi)
eucl_dist_cross = euclidean_distance_manual(vec_en, vec_vi)
dot_prod_cross = dot_product_manual(vec_en, vec_vi)

print("=" * 70)
print("SO SÁNH CROSS-LANGUAGE: TIẾNG ANH vs TIẾNG VIỆT")
print("=" * 70)
print(f"\nEnglish: '{sentences_en[0]}'")
print(f"Vietnamese: '{sentences_vi[0]}'")
print(f"\n📊 Kết quả:")
print(f"   - Cosine Similarity: {cos_sim_cross:.4f}")
print(f"   - Euclidean Distance: {eucl_dist_cross:.4f}")
print(f"   - Dot Product: {dot_prod_cross:.4f}")

SO SÁNH CROSS-LANGUAGE: TIẾNG ANH vs TIẾNG VIỆT

English: 'I love programming'
Vietnamese: 'Tôi yêu lập trình'

📊 Kết quả:
   - Cosine Similarity: 0.8762
   - Euclidean Distance: 2.0633
   - Dot Product: 13.4091

💡 Model multilingual nhận ra được nghĩa tương tự giữa 2 ngôn ngữ!


## 🎯 Phần 7: Ứng dụng Thực Tế - Semantic Search

Tìm câu trả lời phù hợp nhất cho câu hỏi của người dùng.

In [24]:
# Knowledge base (cơ sở tri thức)
knowledge_base = [
    "Python là ngôn ngữ lập trình phổ biến cho AI và Machine Learning",
    "TensorFlow là framework deep learning của Google",
    "PyTorch được phát triển bởi Meta (Facebook)",
    "Jupyter Notebook là công cụ tuyệt vời để phân tích dữ liệu",
    "Pandas giúp xử lý và phân tích dữ liệu dễ dàng",
    "NumPy là thư viện cơ bản cho tính toán khoa học",
    "Matplotlib dùng để vẽ biểu đồ và visualization",
    "Scikit-learn cung cấp các thuật toán machine learning",
]

# Tạo embeddings cho knowledge base
kb_embeddings = model.encode(knowledge_base)

def semantic_search(query, knowledge_base, kb_embeddings, top_k=3, method='cosine'):
    """
    Tìm kiếm semantic trong knowledge base
    
    Args:
        query: Câu hỏi của người dùng
        knowledge_base: Danh sách các câu trong knowledge base
        kb_embeddings: Embeddings của knowledge base
        top_k: Số kết quả trả về
        method: Phương pháp đo độ tương đồng ('cosine', 'euclidean', 'dot')
    
    Returns:
        List of (index, score, text) tuples
    """
    # Tạo embedding cho query
    query_embedding = model.encode([query])[0]
    
    # Tính độ tương đồng
    if method == 'cosine':
        scores = [cosine_similarity_manual(query_embedding, kb_emb) for kb_emb in kb_embeddings]
        reverse = True  # Cosine: càng cao càng tốt
    elif method == 'euclidean':
        scores = [euclidean_distance_manual(query_embedding, kb_emb) for kb_emb in kb_embeddings]
        reverse = False  # Euclidean: càng thấp càng tốt
    elif method == 'dot':
        scores = [dot_product_manual(query_embedding, kb_emb) for kb_emb in kb_embeddings]
        reverse = True  # Dot product: càng cao càng tốt
    else:
        raise ValueError(f"Unknown method: {method}")
    
    # Sắp xếp và lấy top_k
    indexed_scores = list(enumerate(scores))
    indexed_scores.sort(key=lambda x: x[1], reverse=reverse)
    
    results = []
    for idx, score in indexed_scores[:top_k]:
        results.append((idx, score, knowledge_base[idx]))
    
    return results

# Test semantic search
query = "công cụ nào để phân tích dữ liệu?"

print("=" * 70)
print("SEMANTIC SEARCH")
print("=" * 70)
print(f"\n❓ Query: {query}\n")

for method in ['cosine', 'euclidean', 'dot']:
    print(f"\n{'='*70}")
    print(f"Phương pháp: {method.upper()}")
    print(f"{'='*70}")
    
    results = semantic_search(query, knowledge_base, kb_embeddings, top_k=3, method=method)
    
    for rank, (idx, score, text) in enumerate(results, 1):
        print(f"\nTop {rank}: (Score: {score:.4f})")
        print(f"   {text}")

SEMANTIC SEARCH

❓ Query: công cụ nào để phân tích dữ liệu?


Phương pháp: COSINE

Top 1: (Score: 0.5584)
   Jupyter Notebook là công cụ tuyệt vời để phân tích dữ liệu

Top 2: (Score: 0.4728)
   Pandas giúp xử lý và phân tích dữ liệu dễ dàng

Top 3: (Score: 0.3378)
   NumPy là thư viện cơ bản cho tính toán khoa học

Phương pháp: EUCLIDEAN

Top 1: (Score: 3.8378)
   Jupyter Notebook là công cụ tuyệt vời để phân tích dữ liệu

Top 2: (Score: 4.7059)
   NumPy là thư viện cơ bản cho tính toán khoa học

Top 3: (Score: 4.7998)
   Pandas giúp xử lý và phân tích dữ liệu dễ dàng

Phương pháp: DOT

Top 1: (Score: 9.9752)
   Pandas giúp xử lý và phân tích dữ liệu dễ dàng

Top 2: (Score: 9.2938)
   Jupyter Notebook là công cụ tuyệt vời để phân tích dữ liệu

Top 3: (Score: 6.3901)
   Matplotlib dùng để vẽ biểu đồ và visualization


## 📋 Phần 8: Bảng So Sánh Tổng Quan

| Tiêu chí | Cosine Similarity | Euclidean Distance | Dot Product |
|----------|-------------------|-----------------------|-------------|
| **Range** | [-1, 1] | [0, ∞] | (-∞, ∞) |
| **Tốt nhất khi** | 1 | 0 | Càng lớn càng tốt |
| **Phụ thuộc magnitude** | ❌ Không | ✅ Có | ✅ Có |
| **Tốc độ tính toán** | Trung bình | Trung bình | Nhanh nhất |
| **Ứng dụng phổ biến** | NLP, Recommendation | Computer Vision, Clustering | Ranking (với normalized vectors) |
| **Ưu điểm** | Không phụ thuộc scale, ổn định | Trực quan, đơn giản | Nhanh, hiệu quả |
| **Nhược điểm** | Tính toán phức tạp hơn | Nhạy cảm với scale | Phụ thuộc magnitude |
| **Khi nào dùng** | So sánh hướng/góc | So sánh khoảng cách thực | Vectors đã normalize |

## 🎯 Phần 9: BÀI TẬP THỰC HÀNH

### Bài tập 1: Tạo embeddings và tính độ tương đồng
Cho 3 câu sau (tiếng Việt hoặc tiếng Anh tùy chọn), hãy:
1. Tạo embeddings
2. Tính độ tương đồng giữa các cặp câu bằng cả 3 phương pháp
3. Phân tích kết quả

In [ ]:
# TODO: Điền 3 câu của bạn vào đây
my_sentences = [
    "Câu 1 của bạn...",
    "Câu 2 của bạn...",
    "Câu 3 của bạn..."
]

# TODO: Tạo embeddings


# TODO: Tính cosine similarity giữa câu 1 và 2


# TODO: Tính euclidean distance giữa câu 1 và 3


# TODO: Tính dot product giữa câu 2 và 3


# TODO: In kết quả và phân tích

### Bài tập 2: So sánh các phương pháp
Viết một hàm so sánh cả 3 phương pháp đo độ tương đồng cho 2 câu bất kỳ và trả về kết quả dưới dạng DataFrame.

In [ ]:
# TODO: Hoàn thiện hàm này
def compare_similarity_methods(sentence1, sentence2, model):
    # \"\"\"
    # So sánh 3 phương pháp đo độ tương đồng
    
    # Returns:
    #     DataFrame với kết quả của cả 3 phương pháp
    # \"\"\"
    # TODO: Tạo embeddings cho 2 câu
    
    
    # TODO: Tính toán với 3 phương pháp
    
    
    # TODO: Tạo DataFrame và return
    results = {
        'Method': ['Cosine Similarity', 'Euclidean Distance', 'Dot Product'],
        'Score': [0.0, 0.0, 0.0],  # Thay bằng giá trị thực
    }
    
    return pd.DataFrame(results)

# Test hàm của bạn
sentence1 = "Học máy là một lĩnh vực của AI"
sentence2 = "Machine learning is a field of artificial intelligence"

# TODO: Gọi hàm và in kết quả
# df_result = compare_similarity_methods(sentence1, sentence2, model)
# print(df_result)

### Bài tập 3: Xây dựng hệ thống tìm kiếm câu hỏi-đáp
Tạo một knowledge base về một chủ đề bạn quan tâm (ít nhất 10 câu), sau đó:
1. Tạo embeddings cho tất cả câu
2. Cho người dùng nhập câu hỏi
3. Tìm top 3 câu trả lời phù hợp nhất
4. So sánh kết quả với 3 phương pháp khác nhau

In [ ]:
# TODO: Tạo knowledge base của bạn về một chủ đề (ít nhất 10 câu)
my_knowledge_base = [
    "Câu 1 về chủ đề của bạn...",
    "Câu 2...",
    # Thêm ít nhất 8 câu nữa
]

# TODO: Tạo embeddings cho knowledge base


# TODO: Nhập câu hỏi của người dùng
user_query = "Câu hỏi của bạn..."

# TODO: Tìm kiếm với cả 3 phương pháp và so sánh kết quả
# Sử dụng hàm semantic_search đã được định nghĩa ở trên

### Bài tập 4: So sánh các Embedding Models
Thử nghiệm với các embedding models khác nhau và so sánh kết quả:
- `paraphrase-multilingual-MiniLM-L12-v2` (đã dùng)
- `all-MiniLM-L6-v2` (English only, faster)
- `paraphrase-multilingual-mpnet-base-v2` (better quality, slower)

In [ ]:
# TODO: Load các models khác nhau
# model1 = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
# model2 = SentenceTransformer('all-MiniLM-L6-v2')
# model3 = SentenceTransformer('paraphrase-multilingual-mpnet-base-v2')

# TODO: Chọn 2 câu để so sánh
sentence_1 = "Artificial intelligence is the future"
sentence_2 = "AI will shape our tomorrow"

# TODO: Tạo embeddings với mỗi model


# TODO: So sánh cosine similarity từ các models khác nhau


# TODO: So sánh tốc độ, chất lượng, kích thước vector

## 📝 Phần 10: Tổng kết

### Những điều cần nhớ:

1. **Vector Embeddings** biểu diễn ngữ nghĩa trong không gian số
   - Câu có nghĩa giống nhau → vectors gần nhau
   - Cho phép tính toán độ tương đồng ngữ nghĩa

2. **Cosine Similarity** - Phổ biến nhất
   - Đo góc giữa 2 vectors
   - Range: [-1, 1]
   - Không phụ thuộc vào magnitude
   - Tốt nhất cho NLP

3. **Euclidean Distance** - Khoảng cách thực
   - Đo khoảng cách trực tiếp
   - Range: [0, ∞]
   - Phụ thuộc magnitude
   - Tốt cho dữ liệu không gian

4. **Dot Product** - Nhanh nhất
   - Tích vô hướng của 2 vectors
   - Range: (-∞, ∞)
   - Tương đương cosine khi vectors đã normalize
   - Tốt cho ranking

5. **Khi nào dùng phương pháp nào?**
   - **Cosine**: So sánh văn bản, recommendation, search
   - **Euclidean**: Clustering, computer vision, spatial data
   - **Dot Product**: Ranking với normalized vectors, neural networks

### Ứng dụng thực tế:
- 🔍 Semantic Search
- 📚 Document Similarity
- 🤖 Chatbots & Q&A Systems
- 🎯 Recommendation Systems
- 📊 Text Clustering & Classification
- 🌐 Cross-lingual Search
- 🔄 Duplicate Detection

## 🔗 Phần 11: Tài nguyên tham khảo

### Thư viện & Models:
- [Sentence Transformers](https://www.sbert.net/)
- [Hugging Face Models](https://huggingface.co/models)
- [OpenAI Embeddings](https://platform.openai.com/docs/guides/embeddings)

### Bài viết & Tutorial:
- [Understanding Vector Embeddings](https://www.pinecone.io/learn/vector-embeddings/)
- [Cosine Similarity Explained](https://www.machinelearningplus.com/nlp/cosine-similarity/)
- [Semantic Search with Embeddings](https://www.sbert.net/examples/applications/semantic-search/README.html)

### Vector Databases:
- Pinecone
- Weaviate
- Milvus
- Qdrant
- ChromaDB

---
**Chúc các bạn học tốt! 🎓**

*Nếu có thắc mắc, hãy thử nghiệm thêm với các ví dụ khác nhau trong các bài tập!*